# Colab Preprocess: OpenFake subset → 256×256 split dataset + EDA

This notebook is designed for your current team setup:

- Raw file on Google Drive: `openfake_subset.zip`
- Split file on Google Drive: `model_split_metadata.csv`
- Split file format: `filename, split, label`
- Output image size: **256×256**
- Output folders:
  - `train/real`, `train/fake`
  - `val/real`, `val/fake`
  - `test/real`, `test/fake`

It saves the processed dataset, metadata, EDA tables, EDA plots, and downloadable ZIP archives back to Google Drive.

> Important: If the files are only under **Shared with me**, add the shared folder as a **shortcut to My Drive** first. Then the default path below should work:
>
> `/content/drive/MyDrive/Image Processing Project/`

In [ ]:
# Install dependencies
!pip -q install pandas pillow tqdm matplotlib

import os
import json
import shutil
import zipfile
import hashlib
from pathlib import Path
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from tqdm.auto import tqdm

try:
    from google.colab import drive, files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

SEED = 42

# Image preprocessing to match teammates
TARGET_IMAGE_SIZE = (256, 256)
OUTPUT_IMAGE_FORMAT = "JPEG"
OUTPUT_IMAGE_SUFFIX = ".jpg"
JPEG_QUALITY = 95

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}

print("IN_COLAB:", IN_COLAB)

## 1. Mount Google Drive

In [ ]:
if IN_COLAB:
    drive.mount("/content/drive")
else:
    print("Not running in Colab; skipping Drive mount.")

## 2. Configure paths

Edit `PROJECT_DIR` if your shared folder shortcut has a different name.

Expected input files:

```text
PROJECT_DIR/
  openfake_subset.zip
  model_split_metadata.csv
```

In [ ]:
# ===== EDIT THIS IF NEEDED =====
PROJECT_DIR = Path("/content/drive/MyDrive/Image Processing Project")

ZIP_PATH = PROJECT_DIR / "openfake_subset.zip"
SPLIT_CSV_PATH = PROJECT_DIR / "model_split_metadata.csv"

# Outputs will be saved here
OUTPUT_ROOT_DRIVE = PROJECT_DIR / "preprocessed_256_outputs"
DRIVE_DATASET_DIR = OUTPUT_ROOT_DRIVE / "openfake_split_256"
DRIVE_ARCHIVES_DIR = OUTPUT_ROOT_DRIVE / "archives"
DRIVE_EDA_DIR = OUTPUT_ROOT_DRIVE / "eda"

# Local runtime paths. Local disk is faster than Drive for processing.
LOCAL_WORK_DIR = Path("/content/openfake_work")
LOCAL_EXTRACT_DIR = LOCAL_WORK_DIR / "dataset"
LOCAL_OUTPUT_DIR = LOCAL_WORK_DIR / "openfake_split_256"

# If True, rebuild output even if it already exists on Drive.
FORCE_REBUILD = False

# If True, extract the raw ZIP again even if a local extracted folder exists.
FORCE_REEXTRACT = False

for p in [OUTPUT_ROOT_DRIVE, DRIVE_ARCHIVES_DIR, DRIVE_EDA_DIR, LOCAL_WORK_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("ZIP_PATH:", ZIP_PATH, "exists:", ZIP_PATH.exists())
print("SPLIT_CSV_PATH:", SPLIT_CSV_PATH, "exists:", SPLIT_CSV_PATH.exists())
print("DRIVE_DATASET_DIR:", DRIVE_DATASET_DIR)

assert ZIP_PATH.exists(), f"Missing zip: {ZIP_PATH}. Add the shared folder as a shortcut to My Drive or update PROJECT_DIR."
assert SPLIT_CSV_PATH.exists(), f"Missing split CSV: {SPLIT_CSV_PATH}. Add the shared folder as a shortcut to My Drive or update PROJECT_DIR."

## 3. Optional: skip rebuild if processed dataset already exists

This prevents repeating the whole preprocessing step if you already produced the dataset once.

In [ ]:
expected_metadata = DRIVE_DATASET_DIR / "metadata" / "split_metadata.csv"

if expected_metadata.exists() and not FORCE_REBUILD:
    print("Processed dataset already exists on Drive:")
    print(DRIVE_DATASET_DIR)
    print("\nSet FORCE_REBUILD=True if you want to regenerate it.")
    SKIP_PROCESSING = True
else:
    SKIP_PROCESSING = False
    print("Processing will run.")

## 4. Extract raw ZIP locally

The notebook extracts to Colab local disk for speed. This does not modify the original ZIP in Drive.

In [ ]:
if not SKIP_PROCESSING:
    if FORCE_REEXTRACT and LOCAL_EXTRACT_DIR.exists():
        shutil.rmtree(LOCAL_EXTRACT_DIR)

    if not LOCAL_EXTRACT_DIR.exists():
        LOCAL_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
        print("Extracting ZIP to local runtime. This may take a few minutes...")
        with zipfile.ZipFile(ZIP_PATH, "r") as z:
            z.extractall(LOCAL_EXTRACT_DIR)
        print("Extraction complete.")
    else:
        print("Local extracted folder already exists; skipping extraction.")

    print("\nExtracted folder preview:")
    for root, dirs, files_ in os.walk(LOCAL_EXTRACT_DIR):
        root_path = Path(root)
        depth = len(root_path.relative_to(LOCAL_EXTRACT_DIR).parts)
        if depth > 2:
            continue
        indent = "  " * depth
        print(f"{indent}{root_path.name}/")
        if files_:
            print(f"{indent}  files: {len(files_)}")
else:
    print("Skipping extraction because processed dataset already exists.")

## 5. Locate raw images and metadata

The split CSV provides the exact files and split labels. If raw `metadata.csv` exists inside the ZIP, this notebook also merges it for better EDA.

In [ ]:
if not SKIP_PROCESSING:
    # Locate images directory
    image_dirs = [p for p in LOCAL_EXTRACT_DIR.rglob("*") if p.is_dir() and p.name.lower() == "images"]
    if image_dirs:
        IMAGES_DIR = image_dirs[0]
    else:
        IMAGES_DIR = LOCAL_EXTRACT_DIR / "images"

    assert IMAGES_DIR.exists(), f"Images directory not found. Expected something like {LOCAL_EXTRACT_DIR}/images"

    # Split metadata from teammate
    split_df = pd.read_csv(SPLIT_CSV_PATH)
    required_cols = {"filename", "split", "label"}
    missing = required_cols - set(split_df.columns)
    assert not missing, f"Split CSV missing columns: {missing}. Found: {list(split_df.columns)}"

    split_df["filename"] = split_df["filename"].astype(str)
    split_df["split"] = split_df["split"].astype(str).str.lower().str.strip()
    split_df["label"] = split_df["label"].astype(str).str.lower().str.strip()

    print("Split CSV shape:", split_df.shape)
    display(split_df.head())
    display(pd.crosstab(split_df["split"], split_df["label"]))

    # Raw metadata inside zip, if present
    raw_metadata_candidates = sorted(LOCAL_EXTRACT_DIR.rglob("metadata.csv"))
    RAW_METADATA_PATH = raw_metadata_candidates[0] if raw_metadata_candidates else None
    print("RAW_METADATA_PATH:", RAW_METADATA_PATH)

    if RAW_METADATA_PATH is not None:
        raw_meta = pd.read_csv(RAW_METADATA_PATH)

        # Try to derive filename from raw metadata
        candidate_path_cols = ["image_path", "path", "filepath", "file_path", "filename", "image"]
        raw_path_col = next((c for c in candidate_path_cols if c in raw_meta.columns), None)

        if raw_path_col is not None:
            raw_meta["filename"] = raw_meta[raw_path_col].astype(str).apply(lambda x: Path(x).name)
            # Avoid duplicate filename rows if present
            raw_meta = raw_meta.drop_duplicates(subset=["filename"], keep="first")
            enriched_df = split_df.merge(raw_meta, on="filename", how="left", suffixes=("", "_raw"))
            print("Merged split CSV with raw metadata.")
        else:
            enriched_df = split_df.copy()
            print("Raw metadata found but no image path/filename column recognized.")
    else:
        enriched_df = split_df.copy()
        print("No raw metadata.csv found inside ZIP.")

    print("Enriched shape:", enriched_df.shape)
    display(enriched_df.head())

else:
    print("Skipping metadata loading because processed dataset already exists.")

## 6. Save resized 256×256 images according to `model_split_metadata.csv`

This uses the exact team split from `model_split_metadata.csv`.

In [ ]:
if not SKIP_PROCESSING:
    if LOCAL_OUTPUT_DIR.exists():
        shutil.rmtree(LOCAL_OUTPUT_DIR)

    for split_name in ["train", "val", "test"]:
        for label_name in ["real", "fake"]:
            (LOCAL_OUTPUT_DIR / split_name / label_name).mkdir(parents=True, exist_ok=True)

    LOCAL_METADATA_DIR = LOCAL_OUTPUT_DIR / "metadata"
    LOCAL_EDA_DIR = LOCAL_OUTPUT_DIR / "eda"
    LOCAL_PLOTS_DIR = LOCAL_EDA_DIR / "plots"

    for p in [LOCAL_METADATA_DIR, LOCAL_EDA_DIR, LOCAL_PLOTS_DIR]:
        p.mkdir(parents=True, exist_ok=True)

    def make_safe_filename(src_path, row_index):
        digest = hashlib.sha1(str(src_path).encode("utf-8")).hexdigest()[:12]
        return f"{row_index:07d}_{digest}{OUTPUT_IMAGE_SUFFIX}"

    def save_resized_image(src, dst):
        with Image.open(src) as img:
            img = ImageOps.exif_transpose(img)
            img = img.convert("RGB")
            img = img.resize(TARGET_IMAGE_SIZE, Image.Resampling.LANCZOS)
            img.save(dst, format=OUTPUT_IMAGE_FORMAT, quality=JPEG_QUALITY, optimize=True)

    saved_rows = []
    missing_rows = []
    bad_rows = []

    for idx, row in tqdm(enriched_df.iterrows(), total=len(enriched_df), desc="Saving resized images"):
        filename = str(row["filename"])
        split_name = str(row["split"]).lower().strip()
        label_name = str(row["label"]).lower().strip()

        if split_name not in {"train", "val", "test"}:
            bad_rows.append({"filename": filename, "error": f"Invalid split: {split_name}"})
            continue

        if label_name not in {"real", "fake"}:
            bad_rows.append({"filename": filename, "error": f"Invalid label: {label_name}"})
            continue

        src = IMAGES_DIR / filename
        if not src.exists():
            missing_rows.append({"filename": filename, "expected_path": str(src), "split": split_name, "label": label_name})
            continue

        safe_name = make_safe_filename(src, idx)
        rel_path = Path(split_name) / label_name / safe_name
        dst = LOCAL_OUTPUT_DIR / rel_path

        try:
            save_resized_image(src, dst)
        except Exception as e:
            bad_rows.append({"filename": filename, "expected_path": str(src), "error": repr(e)})
            continue

        out = row.to_dict()
        out["original_filename"] = filename
        out["saved_filename"] = safe_name
        out["saved_relative_path"] = str(rel_path)
        out["saved_absolute_path"] = str(dst)
        out["saved_width"] = TARGET_IMAGE_SIZE[0]
        out["saved_height"] = TARGET_IMAGE_SIZE[1]
        out["preprocessing"] = "exif_transpose_rgb_resize_256_jpeg95"
        saved_rows.append(out)

    saved_df = pd.DataFrame(saved_rows)
    missing_df = pd.DataFrame(missing_rows)
    bad_df = pd.DataFrame(bad_rows)

    print("Saved:", len(saved_df))
    print("Missing:", len(missing_df))
    print("Bad:", len(bad_df))

    display(pd.crosstab(saved_df["split"], saved_df["label"]))

    saved_df.to_csv(LOCAL_METADATA_DIR / "split_metadata.csv", index=False)
    pd.crosstab(saved_df["split"], saved_df["label"]).to_csv(LOCAL_METADATA_DIR / "split_summary.csv")

    if len(missing_df):
        missing_df.to_csv(LOCAL_METADATA_DIR / "missing_images.csv", index=False)
    if len(bad_df):
        bad_df.to_csv(LOCAL_METADATA_DIR / "bad_images.csv", index=False)

else:
    print("Skipping image save because processed dataset already exists.")

## 7. EDA tables and plots

The EDA is saved inside both:

```text
openfake_split_256/eda/
preprocessed_256_outputs/eda/
```

In [ ]:
if not SKIP_PROCESSING:
    # Basic split/class counts
    class_counts = saved_df.groupby(["split", "label"]).size().reset_index(name="count")
    class_counts.to_csv(LOCAL_EDA_DIR / "class_counts.csv", index=False)
    display(class_counts)

    # Model/source EDA if columns are available
    possible_model_cols = [c for c in ["model", "source", "type", "release_date"] if c in saved_df.columns]
    if possible_model_cols:
        for col in possible_model_cols:
            counts = (
                saved_df.groupby(["split", "label", col], dropna=False)
                .size()
                .reset_index(name="count")
                .sort_values(["split", "label", "count"], ascending=[True, True, False])
            )
            counts.to_csv(LOCAL_EDA_DIR / f"{col}_counts.csv", index=False)
            print(f"Top {col} counts:")
            display(counts.head(20))

    # Prompt length EDA if available
    if "prompt" in saved_df.columns:
        saved_df["prompt_length"] = saved_df["prompt"].fillna("").astype(str).str.len()
        prompt_stats = saved_df.groupby("split")["prompt_length"].describe()
        prompt_stats.to_csv(LOCAL_EDA_DIR / "prompt_length_stats.csv")
        display(prompt_stats)

    def save_plot(path):
        plt.tight_layout()
        plt.savefig(path, dpi=160)
        plt.show()

    # Plot: images per split
    plt.figure(figsize=(7, 4))
    saved_df["split"].value_counts().reindex(["train", "val", "test"]).plot(kind="bar")
    plt.title("Images per split")
    plt.xlabel("Split")
    plt.ylabel("Count")
    plt.xticks(rotation=0)
    save_plot(LOCAL_PLOTS_DIR / "images_per_split.png")

    # Plot: class balance
    pivot = saved_df.pivot_table(index="split", columns="label", values="saved_relative_path", aggfunc="count", fill_value=0)
    pivot = pivot.reindex(["train", "val", "test"])
    pivot.plot(kind="bar", figsize=(7, 4))
    plt.title("Real/fake balance by split")
    plt.xlabel("Split")
    plt.ylabel("Count")
    plt.xticks(rotation=0)
    save_plot(LOCAL_PLOTS_DIR / "class_balance_by_split.png")

    # Plot: model counts if available
    if "model" in saved_df.columns:
        for split_name in ["train", "val", "test"]:
            plot_df = saved_df[saved_df["split"] == split_name]
            counts = plot_df["model"].fillna("unknown").value_counts()
            plt.figure(figsize=(11, 5))
            counts.plot(kind="bar")
            plt.title(f"Model/source counts — {split_name}")
            plt.xlabel("Model/source")
            plt.ylabel("Count")
            plt.xticks(rotation=60, ha="right")
            save_plot(LOCAL_PLOTS_DIR / f"model_source_counts_{split_name}.png")

    # Plot: prompt lengths if available
    if "prompt_length" in saved_df.columns:
        plt.figure(figsize=(8, 4))
        saved_df["prompt_length"].clip(upper=1000).hist(bins=50)
        plt.title("Prompt length distribution, clipped at 1000 chars")
        plt.xlabel("Prompt length")
        plt.ylabel("Count")
        save_plot(LOCAL_PLOTS_DIR / "prompt_length_distribution.png")

else:
    print("Skipping EDA generation because processed dataset already exists.")

## 8. Verify saved image size

In [ ]:
if not SKIP_PROCESSING:
    saved_sizes = Counter()
    checked = 0

    for p in LOCAL_OUTPUT_DIR.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS:
            with Image.open(p) as img:
                saved_sizes[img.size] += 1
            checked += 1

    print("Checked images:", checked)
    print("Most common sizes:", saved_sizes.most_common(10))
    assert saved_sizes.get(TARGET_IMAGE_SIZE, 0) == checked, "Some saved images are not 256x256."

else:
    print("Skipping size verification because processed dataset already exists.")

## 9. Save run config and copy processed dataset to Google Drive

In [ ]:
if not SKIP_PROCESSING:
    run_config = {
        "seed": SEED,
        "target_image_size": TARGET_IMAGE_SIZE,
        "output_image_format": OUTPUT_IMAGE_FORMAT,
        "jpeg_quality": JPEG_QUALITY,
        "project_dir": str(PROJECT_DIR),
        "zip_path": str(ZIP_PATH),
        "split_csv_path": str(SPLIT_CSV_PATH),
        "images_dir": str(IMAGES_DIR),
        "local_output_dir": str(LOCAL_OUTPUT_DIR),
        "drive_dataset_dir": str(DRIVE_DATASET_DIR),
        "saved_images": int(len(saved_df)),
        "missing_images": int(len(missing_df)),
        "bad_images": int(len(bad_df)),
        "notes": "Uses teammate-provided model_split_metadata.csv. Resizes images to 256x256 and saves train/val/test folders.",
    }

    with open(LOCAL_METADATA_DIR / "run_config.json", "w") as f:
        json.dump(run_config, f, indent=2)

    if DRIVE_DATASET_DIR.exists():
        shutil.rmtree(DRIVE_DATASET_DIR)

    print("Copying processed dataset folder to Drive. This can take time...")
    shutil.copytree(LOCAL_OUTPUT_DIR, DRIVE_DATASET_DIR)

    # Also copy EDA to the top-level EDA folder for easy access
    if DRIVE_EDA_DIR.exists():
        shutil.rmtree(DRIVE_EDA_DIR)
    shutil.copytree(LOCAL_EDA_DIR, DRIVE_EDA_DIR)

    print("Saved processed dataset to:", DRIVE_DATASET_DIR)
    print("Saved EDA to:", DRIVE_EDA_DIR)

else:
    print("Processed dataset already exists here:", DRIVE_DATASET_DIR)

## 10. Create downloadable archives

For large datasets, direct browser download can fail. The safest option is downloading the ZIPs from Google Drive.

This cell creates:

- one full archive
- separate train/val/test archives
- one metadata + EDA archive

In [ ]:
CREATE_FULL_ZIP = True
CREATE_SPLIT_ZIPS = True
CREATE_METADATA_EDA_ZIP = True

# Use the Drive dataset if processing was skipped, otherwise local output.
SOURCE_FOR_ARCHIVE = DRIVE_DATASET_DIR if SKIP_PROCESSING else LOCAL_OUTPUT_DIR

DRIVE_ARCHIVES_DIR.mkdir(parents=True, exist_ok=True)

def make_zip_archive(source_dir, output_zip_path):
    source_dir = Path(source_dir)
    output_zip_path = Path(output_zip_path)
    if output_zip_path.exists():
        output_zip_path.unlink()
    base_name = str(output_zip_path.with_suffix(""))
    archive_path = shutil.make_archive(base_name, "zip", source_dir)
    return Path(archive_path)

created = []

if CREATE_FULL_ZIP:
    full_zip = DRIVE_ARCHIVES_DIR / "openfake_split_256_full.zip"
    archive = make_zip_archive(SOURCE_FOR_ARCHIVE, full_zip)
    created.append(archive)
    print("Created:", archive, "size GB:", round(archive.stat().st_size / (1024**3), 3))

if CREATE_SPLIT_ZIPS:
    for split_name in ["train", "val", "test"]:
        split_source = SOURCE_FOR_ARCHIVE / split_name
        split_zip = DRIVE_ARCHIVES_DIR / f"openfake_split_256_{split_name}.zip"
        archive = make_zip_archive(split_source, split_zip)
        created.append(archive)
        print("Created:", archive, "size GB:", round(archive.stat().st_size / (1024**3), 3))

if CREATE_METADATA_EDA_ZIP:
    package_dir = LOCAL_WORK_DIR / "metadata_eda_package"
    if package_dir.exists():
        shutil.rmtree(package_dir)
    package_dir.mkdir(parents=True, exist_ok=True)

    if (SOURCE_FOR_ARCHIVE / "metadata").exists():
        shutil.copytree(SOURCE_FOR_ARCHIVE / "metadata", package_dir / "metadata")
    if (SOURCE_FOR_ARCHIVE / "eda").exists():
        shutil.copytree(SOURCE_FOR_ARCHIVE / "eda", package_dir / "eda")

    meta_zip = DRIVE_ARCHIVES_DIR / "openfake_split_256_metadata_eda.zip"
    archive = make_zip_archive(package_dir, meta_zip)
    created.append(archive)
    print("Created:", archive, "size GB:", round(archive.stat().st_size / (1024**3), 3))

print("\nArchives saved in:", DRIVE_ARCHIVES_DIR)
for p in sorted(DRIVE_ARCHIVES_DIR.glob("*.zip")):
    print(" -", p.name, "GB:", round(p.stat().st_size / (1024**3), 3))

## 11. Optional direct download from Colab

Use this for the metadata/EDA ZIP, or small split ZIPs. For the full dataset ZIP, downloading from Google Drive is usually more reliable.

In [ ]:
DIRECT_DOWNLOAD_METADATA_EDA = False
DIRECT_DOWNLOAD_FULL_ZIP = False

if IN_COLAB and DIRECT_DOWNLOAD_METADATA_EDA:
    meta_zip = DRIVE_ARCHIVES_DIR / "openfake_split_256_metadata_eda.zip"
    assert meta_zip.exists(), f"Missing {meta_zip}"
    files.download(str(meta_zip))

if IN_COLAB and DIRECT_DOWNLOAD_FULL_ZIP:
    full_zip = DRIVE_ARCHIVES_DIR / "openfake_split_256_full.zip"
    assert full_zip.exists(), f"Missing {full_zip}"
    files.download(str(full_zip))

if not DIRECT_DOWNLOAD_METADATA_EDA and not DIRECT_DOWNLOAD_FULL_ZIP:
    print("Direct download disabled. Recommended: download archives from Google Drive.")

## 12. How to use this output on Kaggle

Recommended Kaggle flow:

1. Download `openfake_split_256_full.zip` or the split ZIPs from Drive.
2. Create a Kaggle dataset from the processed output, not from the raw ZIP.
3. In the training notebook, attach the processed Kaggle dataset.

If uploaded extracted, training paths will look like:

```text
/kaggle/input/openfake-split-256/openfake_split_256/train/real
/kaggle/input/openfake-split-256/openfake_split_256/train/fake
/kaggle/input/openfake-split-256/openfake_split_256/val/real
/kaggle/input/openfake-split-256/openfake_split_256/val/fake
/kaggle/input/openfake-split-256/openfake_split_256/test/real
/kaggle/input/openfake-split-256/openfake_split_256/test/fake
```

If your model uses CSV metadata, use:

```text
metadata/split_metadata.csv
```